# 感知 harness 的插件评估



## 概览

一个在你自己的评估中得分很高的插件，一旦在其所发布的产品中运行，仍然可能失败。模型周围的 harness
在很大程度上造成了这种差距，因为它控制着模型看到的提示以及模型能够调用的工具。本文以三种不同的保真度
运行同一组测试用例，并展示如何将差异追溯到导致它的那一层。

模型经常需要训练本身无法提供的最新、专业的数据和操作。一个
[插件](https://developers.openai.com/plugins/concepts/plugins)可以将技能或带有可调用工具的 MCP 服务器
与自然语言指令打包在一起。本文的插件将 Codex harness（现已成为 ChatGPT 桌面应用的一部分）
连接到美国劳工统计局（Bureau of Labor Statistics，BLS）的数据。
由于 [BLS API](https://www.bls.gov/developers/) 需要已知的序列 ID，插件会先将用户的问题
解析为一组排好序的候选序列。

对于这样的插件，最好在多个层级上进行评估，以在速度、成本和保真度之间取得平衡。同一个模型可能
在一个小型、受控的循环中正确解析每一个有歧义的查询，而一旦在真实的智能体中运行，却几乎不调用工具。
一种设置下得到的分数无法迁移到另一种设置，因为
它衡量设置本身的程度不亚于衡量模型。

| 方法                           | 测试内容                                                                                       | 遗漏内容                                                                                                                                 | 所需条件                                                                                            | 成本     | 保真度                                         |
| ---------------------------------- | --------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------- | -------- | ------------------------------------------------ |
| **直接工具测试**         | 将结构化输入直接发送给工具，非常适合调试工具逻辑                        | 自然语言理解与工具选择                                                                                             | 工具代码和结构化语料                                                                           | 免费     | 最低                                           |
| **独立函数调用循环** | 在基于 MCP 的受控函数调用循环中测试自然语言理解与工具选择   | 产品 harness 的行为                                                                                                                       | 运行中的 MCP 服务器、模型 API 访问权限以及循环代码                                                   | 中等 | 中等                                         |
| **产品 harness**           | 在安装了插件的 [Codex](https://openai.com/codex/) harness 中运行同样的查询 | 评估设置可能无法复现的客户端特定行为。将具有代表性的样本与目标客户端进行比较，以找出差距 | 运行中的 MCP 服务器、已安装的插件、Codex 身份验证（此处使用 API 密钥）以及 harness 配置 | 最高  | 最高，仅次于 ChatGPT 桌面应用本身 |

本文的演练使用一个简化的示例插件，它基于合成数据运行，并有意只覆盖该领域的一部分，
以便暴露评估失败。演练还使用了一组精心挑选的五个评估用例。

结果章节报告的是对生产版 BLS 插件的另一项独立评估，涵盖 56 项 BLS 调查中的 108 个问题。
其代码和数据不随本文一起发布。



## 设置

本文的可运行示例随文发布。配套的 [README](https://github.com/openai/openai-cookbook/blob/main/examples/partners/harness_aware_plugin_evals/README.md) 列出了
前提条件，并详细介绍了每一个步骤，
而 [Makefile](https://github.com/openai/openai-cookbook/blob/main/examples/partners/harness_aware_plugin_evals/Makefile) 则将每个步骤封装为一个独立的 target。

BLS 领域过于庞杂，不适合直接用来讲解评估机制。从这里开始，
本文将基于它的一个微缩版本展开，这个版本小到可以完整阅读，
并能在笔记本电脑上几分钟内运行完毕。



### 依赖

该示例需要 [uv](https://docs.astral.sh/uv/getting-started/installation/)、
[Node.js 及 npm](https://nodejs.org/en/download)，以及一个 OpenAI API 密钥，用于
调用模型的运行。

Python 依赖会根据随附的 lockfile 安装到内核所在的环境中：

In [1]:
!uv sync --active --inexact --no-dev --quiet

将 `.env.example` 复制为 `.env` 并填入密钥。独立循环从 `.env` 中读取配置。
Codex 从 `.env` 中读取模型和密钥，从插件的
`.mcp.json` 中读取服务器 URL。

直接工具测试不使用 `.env`，也不产生任何费用。
另外两种方法需要有效的 OpenAI API 密钥，并会发起付费的模型调用。
在默认设置下，按 API 价格计算，一次运行大约花费 $0.01。
将 `EVAL_MODEL` 设置为你的密钥可以访问的模型。它与结果章节中
比较的模型无关。

由于这两种方法通过 HTTP 调用 MCP 服务器，因此在任一评估开始之前，服务器
都必须已经运行。`make server` 会在前台运行服务器，因此它需要单独占用一个终端。

随本文发布的 notebook 运行时不产生任何费用。直接工具测试会实时运行。
另外两种方法会回放 `data/` 下录制好的运行结果。设置 `RUN_LIVE=1` 后，
独立循环以及不带服务器指令的 Codex 运行会实时执行。
带服务器指令的 Codex 运行则保持为录制结果，除非你
手动复现它。
`notebook_setup.py` 中的 `prepare()` 会加载 `.env`，并将 Codex 指向本示例。在
实时模式下，它还会检查密钥、清空 `outputs/`、写入 Codex 配置，并在需要时
启动服务器：

In [2]:
from notebook_setup import prepare, stop_server

RUN_LIVE = prepare()

[Promptfoo](https://www.promptfoo.dev/) 是一个开源的评估运行器。它从
YAML 文件中加载测试用例，将每个用例发送给一个 provider（负责回答它的模型或脚本），并用断言
对输出进行评分。在本示例中，它同时运行独立循环和 Codex，其配置文件
包含三种方法共享的测试用例。根据 lockfile 安装 Node 包，其中包括 promptfoo：

In [3]:
if RUN_LIVE:
    !npm ci --silent

### 示例插件

本文严格区分以下四个术语：

- 工具（Tool）：暴露给模型的单个函数，由其参数的 schema 定义。
- MCP 服务器：实现 Model Context Protocol 并向客户端发布可调用工具的
  进程。
- 插件（Plugin）：将上述部分与产品安装所需的元数据打包在一起的软件包。
- Harness：运行时模型周围的一切，包括智能体循环、提示
  格式化、工具调用执行、轮次限制和沙箱策略。

服务器暴露两个工具，二者都定义在 `server.py` 中，并通过
[FastMCP](https://gofastmcp.com/) 以 HTTP 方式提供服务。`resolve(indicators)` 接收一个或多个经济指标，
并返回一个排好序的候选 BLS 序列列表，每个序列都附带置信度分数。
`fetch_bls_data(series_ids)` 接收序列 ID 并返回其观测值；对于目录中不存在的 ID，
则返回一条错误记录。

这两个工具背后是合成的 fixture 数据，旨在以微缩形式复现真实领域的三个
特性（多个有效答案中存在一个默认答案、真正的歧义，以及数据
缺口），并有意不包含 CPI。这些序列 ID 看起来像真实的 BLS 代码，
例如 `LNS14000000` 和 `CES0000000001`，但它们本身并没有内部结构。三个
指标分别代表这三种特性：

- `unemployment rate` 带有刻意设计的歧义。`resolve` 会返回经季节性调整的
  序列和未经调整的序列，并按排名给出，其中调整后的版本是惯常的默认选项，
  最终选择则留给调用方。
- `employment` 返回两个衡量不同内容的候选项，且置信度相同：
  住户调查的就业水平和非农就业人数，没有可以退而采用的默认项。
- `consumer price index` 没有匹配项。解析器返回一个空的候选列表，
  并附带一条说明，指出该缺口是确定的。

直接调用 `resolve` 即可一次性看到这三种情况：

In [4]:
from server import IndicatorQuery, resolve

resolve(indicators=[
    IndicatorQuery(indicator="unemployment rate"),
    IndicatorQuery(indicator="employment"),
    IndicatorQuery(indicator="consumer price index"),
])

{'results': [{'indicator': 'unemployment rate',
   'candidates': [{'series_id': 'LNS14000000',
     'title': 'Unemployment Rate - Seasonally Adjusted',
     'confidence': 1.0},
    {'series_id': 'LNU04000000',
     'title': 'Unemployment Rate - Not Seasonally Adjusted',
     'confidence': 0.6}]},
  {'indicator': 'employment',
   'candidates': [{'series_id': 'LNS12000000',
     'title': 'Employment Level - Seasonally Adjusted',
     'confidence': 0.5},
    {'series_id': 'CES0000000001',
     'title': 'All Employees, Total Nonfarm - Seasonally Adjusted',
     'confidence': 0.5}]},
  {'indicator': 'consumer price index',
   'candidates': [],
   'note': 'No series in this catalog covers that indicator. This is a definitive answer, not a transient failure.'}]}

### 示例数据集

示例语料中的每个用例都并列提供两种形式：一个供确定性检查传给解析器的
结构化字段，以及一个模型必须自行转换为相同请求的
自然语言查询。`expected_series_ids` 是共享的期望
字段。空字符串表示该用例
不应获取任何序列。有歧义的用例多一个字段
`expected_top_ids`：解析器应返回两个并列的候选项，而模型应
不获取任何数据并发问。全部五个用例都在同一个文件 `promptfooconfig.yaml` 中，因此
每种方法读取的都是同一份语料。下面展示其中三个：

```yaml
tests:
  - description: unemployment-rate
    vars:
      indicator: unemployment rate
      query: What is the U.S. unemployment rate?
      expected_tools: "resolve,fetch_bls_data"
      expected_series_ids: "LNS14000000"
    metadata:
      mechanism: ambiguous_default
      expected_type: exact

  - description: consumer-price-index
    vars:
      indicator: consumer price index
      query: What is the current consumer price index?
      expected_tools: "resolve"
      expected_series_ids: ""
    metadata:
      mechanism: coverage_gap
      expected_type: gap

  - description: employment-ambiguous
    vars:
      indicator: employment
      query: What is U.S. employment?
      expected_tools: "resolve"
      expected_series_ids: ""
      expected_top_ids: "LNS12000000,CES0000000001"
    metadata:
      mechanism: ambiguous_concept
      expected_type: clarify

```

`mechanism` 标记用例测试的内容，`expected_type` 标记什么才算正确答案。二者
都作为 promptfoo 元数据存储。评分器并不使用它们。它们会显示在运行报告中，
并允许你按这些字段过滤一次运行：

```shell
npx promptfoo eval -c promptfooconfig.yaml --filter-metadata mechanism=coverage_gap
```

借助该过滤器，你无需手动阅读每一行，就能发现诸如“模型只在覆盖缺口上失败”之类的
规律。



## 直接工具测试：没有模型参与的确定性检查

直接工具测试将同一份示例语料的结构化形式交给插件的 `resolve`
工具运行。这里没有需要理解的自然语言问题，也没有由模型来决定调用哪个工具。
该检查加载 `promptfooconfig.yaml`，并直接从 `server.py` 调用 `resolve`，不涉及
MCP 客户端、promptfoo 或网络。

In [5]:
import yaml

from server import IndicatorQuery, resolve

with open("promptfooconfig.yaml") as f:
    CASES = [test["vars"] for test in yaml.safe_load(f)["tests"]]


def top_candidates(indicator: str) -> set[str]:
    result = resolve(indicators=[IndicatorQuery(indicator=indicator)])
    candidates = result["results"][0]["candidates"]
    if not candidates:
        return set()
    best = max(candidate["confidence"] for candidate in candidates)
    return {c["series_id"] for c in candidates if c["confidence"] == best}


top_candidates("consumer price index")

set()

运行器将该集合与每个用例的期望进行比较，并为每个用例打印一行。
consumer price index 用例不期望返回任何序列，因为它不在示例插件识别的四个
指标之中。运行器还会检查 `resolve` 在这种情况下返回的是空的
候选列表，而不是瞎猜。
`level_one.py` 大约一秒钟就能运行完全部五个用例：

In [6]:
%run level_one.py

PASS  'unemployment rate'                 -> LNS14000000
PASS  'labor force participation rate'    -> LNS11300000
PASS  'nonfarm payroll employment'        -> CES0000000001
PASS  'consumer price index'              -> None
PASS  'employment'                        -> CES0000000001, LNS12000000

5/5 passed


直接工具测试的优势包括：

- 快速且免费：不需要 API 密钥、网络调用或模型。
- 确定性：相同的输入在每次运行中都会产生相同的输出，不会有模型带来的
  运行间噪声；并且当某个用例失败时，脚本会以非零状态退出，因此
  可以直接接入 CI。
- 诊断精准：失败直接指向解析器自身的逻辑，不存在
  是模型还是 harness 导致失败的疑问。

直接工具测试并不检查模型能否将通俗语言的问题转换为恰当的工具调用和最终答案。
接下来的两种方法加入了模型，首先是在同样五个用例上运行的独立函数调用循环。



## 独立函数调用循环

独立评估使用一个由应用自己掌控的小型函数调用循环，而不是
像 Codex 这样完整的智能体 harness。模型接收 MCP 工具，选择调用哪些工具，
观察其结果，并生成最终答案。

主要优势包括：

- 成本更低：同样的查询所用的 token 比在完整智能体
  harness 中更少。
- 设置简单：无需安装插件，也没有智能体 home 目录、shell、审批
  策略或容器化的 Codex 运行时。
- 调试直观：没有隐藏行为，只有一个配置文件，失败也
  易于隔离。

只需修改提示和工具名称，这些代码片段就可以适配其他 MCP 服务器。
本示例由 Promptfoo 运行，但其他评估运行器同样可以实现这三种
方法。



### 函数调用循环

该循环使用 [Responses API](https://developers.openai.com/api/reference/responses/overview)。它暴露
MCP 工具，并记录每一次调用以供后续检查。它在 `input` 中携带完整对话并设置 `store: False`，因此不需要
服务器端状态，也能在零数据保留（zero-data-retention）的密钥上运行。

服务器公布的工具就是模型看到的工具：

```python
async with AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"]) as openai, Client(mcp_url) as mcp:
    mcp_tools = await mcp.list_tools()
    tools = [
        {
            "type": "function",
            "name": tool.name,
            "description": tool.description or "",
            "parameters": tool.inputSchema,
        }
        for tool in mcp_tools
    ]
```

每一步都会发送完整对话，而一个不请求任何工具的响应会结束本次运行：

```python
for step in range(max_steps + 1):
    response = await openai.responses.create(input=conversation, **request)
    function_calls = [item for item in response.output if item.type == "function_call"]
    if not function_calls:
        return {
            "answer": response.output_text,
            "tool_calls": recorded_calls,
            "completed": True,
        }
    if step == max_steps:
        break

    # The API rejects round-tripped items that still carry null-valued fields.
    conversation += [item.model_dump(exclude_none=True) for item in response.output]
    for call in function_calls:
        arguments = json.loads(call.arguments or "{}")
        result = await mcp.call_tool(call.name, arguments)
        recorded_calls.append({"name": call.name, "arguments": arguments, "result": result.structured_content})
```

每个工具结果都会作为 `function_call_output` 条目放回对话中，这样下一步
就能看到它。



### 断言

独立循环和 Codex 共享 `eval_grading.py` 中的 `grade`。它检查的是一次运行中的
工具调用，而不是答案文本。当预期的工具都已运行、返回了预期的序列、
没有获取其他任何序列，并且运行以非空答案结束时，用例即通过。
“没有其他序列”这条规则也能捕获与真实 ID 一同被获取的虚构 ID。
对于 `employment-ambiguous`，它会检查是否调用了 `resolve`，且两个候选项都未被
获取。它无法判断该答案是否请用户进行澄清，也无法判断答案是否陈述了
检索到的值。这些检查都不能证明答案是正确的。正式发布的评估还
需要答案层面的检查，例如某个 [promptfoo 断言](https://www.promptfoo.dev/docs/configuration/expected-outputs/)。
演练中五个用例的计数来自这个评分器。生产环境的通过率则来自
另一个单独的 LLM 评分标准评判器。

对一个手写的、虚构了 CPI 序列的运行调用 `grade`，会返回如下结论：

In [7]:
from eval_grading import grade

invented = [{"name": "resolve", "arguments": {"indicators": ["consumer price index"]}, "result": {}},
            {"name": "fetch_bls_data", "arguments": {"series_ids": ["CUUR0000SA0"]}, "result": {}}]
grade(invented, {"expected_tools": "resolve", "expected_series_ids": ""}, "unavailable", completed=True)

{'pass': False,
 'score': 0.0,
 'reason': "fetched series outside the expectation: ['CUUR0000SA0']"}

### 将数据集连接到 provider

数据集就是“设置”部分展示的那个 `promptfooconfig.yaml`，包括其中的 `tests` 块。每个
用例的 `vars.query` 是语料中的自然语言部分，`vars.expected_series_ids`
是共享的判定依据。文件的其余部分将该数据集连接到 provider 和
断言：

```yaml
description: Standalone BLS MCP function-calling evaluation

providers:
  - id: file://provider.py
    label: standalone-mcp
    config:
      model: "{{ env.EVAL_MODEL }}"
      model_reasoning_effort: "{{ env.EVAL_REASONING_EFFORT | default('medium', true) }}"
      mcp_url: "{{ env.MCP_URL }}"

prompts:
  - "{{query}}"

defaultTest:
  assert:
    - type: python
      value: file://assert_result.py
```



### 运行评估

评估只需一次 promptfoo 调用。`--no-cache` 确保
重复运行是真实的模型运行，而不是 promptfoo 的回放；本次运行会将完整 trace 写入
`outputs/l2-results.json`：

In [8]:
if RUN_LIVE:
    !npx promptfoo eval -c promptfooconfig.yaml --no-cache --output outputs/l2-results.json

存在失败用例的运行会以非零状态退出，下面的 Codex 运行正是有意如此，因此结论
应以后面的评分为准。

![独立循环的 Promptfoo Web 报告](../../../images/harness-aware-level-2-web.png)

*独立循环录制的五用例运行的 promptfoo 报告。每一行是一个用例，
Outputs 列包含模型的最终答案以及评分器读取的
已记录工具调用。*

在随附的录制运行中，全部五个用例都通过了，包括 consumer price index 用例。
`resolve` 没有返回任何候选项，而循环的系统提示禁止虚构序列 ID，因此
模型报告了这一缺口。同一个用例在 Codex 下却失败了。



## 产品 harness：Codex

Promptfoo 的 `openai:codex-sdk` provider 将同样的五个查询发送给 Codex。它启动 Codex 时
使用一个已安装 BLS 插件的 Codex home，并添加用户在桌面应用中输入
`@BLS` 时得到的前缀。

这两种设置得到的指令也不同。独立循环的系统提示告诉
模型不要虚构序列 ID，并在候选项并列时说出两者的名称。Codex 则没有得到这样的
指令。



### Codex provider 设置

`config.toml` 通过绝对路径指向 `.codex-home/` marketplace 并启用插件。仓库中
附带 `config.toml.example`。实时运行会写入当前 checkout 的路径，就像 `make setup`
在终端中所做的那样。

同一个文件还定义了一个[权限配置文件（permission profile）](https://developers.openai.com/codex/permissions)，
用于限定一次运行可以访问的范围：

```toml
default_permissions = "eval"

[permissions.eval.filesystem]
":workspace_roots" = "deny"
```

主要的 provider 设置如下：

```yaml
providers:
  - id: openai:codex-sdk
    config:
      model: "{{ env.EVAL_MODEL }}"
      working_dir: .
      enable_streaming: true
      cli_env:
        CODEX_HOME: "{{ env.CODEX_HOME }}"
```

文件的其余部分设置推理 effort，禁用网络访问和网络搜索，关闭
审批和 git 仓库检查，并将 `tests` 指向 `level_3_tests.py`，后者会加载
共享语料。

在项目文件夹内交给智能体一个 shell，会带来严重的评估风险。
`promptfooconfig.yaml` 保存着每个测试用例的预期答案，而 `.env` 保存着真实的
API 密钥。拥有文件访问权限的模型可能会读取它们，并照着答案
文件通过每个用例。权限配置文件可以防止这种情况。`deny` 规则会锁定该目录，禁止
读写，从而使测量始终针对插件本身。评估仍然可以运行，因为
文件系统规则并不适用于通过 HTTP 的 MCP 流量。显式设置的 `CODEX_HOME` 会将 Codex
指向该插件。如果没有它，Codex 会从用户的默认 home 启动，BLS 插件不会
加载，模型也就无法调用 `resolve` 或 `fetch_bls_data`。

提示本身仍保持为自然语言查询。前缀通过
`defaultTest.options.prefix` 传入，下面的文本与用户在桌面应用中添加
`@BLS` 提及时 Codex 看到的内容一致。

```yaml
prompts:
  - "{{query}}"

defaultTest:
  options:
    prefix: "[@BLS](plugin://BLS@cookbook-plugins) "
```

![ChatGPT 桌面应用前缀](../../../images/harness-aware-desktop-prefix.png)

`enable_streaming: true` 让 promptfoo 在 provider 响应中保留 Codex 的条目流。
断言从该 trace 中读取 `mcp_tool_call` 条目，只要其中任何一个失败就判定该用例失败，
并将它们规范化为与独立循环记录相同的
`{"name", "arguments", "result"}` 格式，这样两种方法都会进入
`eval_grading.py` 中的同一个评分器。

```python
mcp_items = [item for item in raw.get("items", []) if item.get("type") == "mcp_tool_call"]
for item in mcp_items:
    if item.get("error") or item.get("status") == "failed":
        return {
            "pass": False,
            "score": 0.0,
            "reason": f"MCP tool call failed: {item.get('tool')}",
        }

calls = [
    {
        "name": item.get("tool"),
        "arguments": _arguments(item),
        "result": (item.get("result") or {}).get("structured_content"),
    }
    for item in mcp_items
]
```

`assert_codex_result.py` 的其余部分从 provider 响应中提取 trace，并在
provider 未报告错误且最终响应非空时，将一次运行视为已完成。



### 运行示例

在填好 `.env` 并启动服务器之后，插件会被安装到同一个 `CODEX_HOME` 中，
评估则针对 `promptfooconfig.codex.yaml` 运行。`CODEX_HOME` 通过“设置”中加载的
环境变量传递给这两条命令：

In [9]:
import subprocess

if RUN_LIVE:
    subprocess.run(["npx", "codex", "plugin", "add", "BLS@cookbook-plugins"], check=True)
    !npx promptfoo eval -c promptfooconfig.codex.yaml --no-cache --output outputs/l3-results.json

安装失败会使该单元格停止运行。否则 Codex 将无法使用任何工具，每个用例都会
因为错误的原因而失败。评分会报告全部五个用例都缺少它们
预期的工具。

![Codex 的 Promptfoo Web 报告](../../../images/harness-aware-level-3-web.png)

*同样五个用例在 Codex harness 中的运行结果，此时服务器尚未发布任何指引。
`consumer-price-index` 失败是因为它获取了一个解析器从未返回的 ID，
`employment-ambiguous` 失败则是因为它获取了解析器确实返回的候选项，而没有询问
用户想要哪一个。*

在 `data/` 下录制的运行中，
独立循环通过了全部五个用例，而 Codex 得分为五分之三。一次新的
实时运行可能会得到不同的计数，因为这两个失败都取决于模型碰巧
选择的路径。

示例服务器中没有 CPI 序列，`resolve` 正确地没有返回
任何内容。随后 Codex 自行提供了一个 ID：它使用 `CUUR0000SA0`（真实的
BLS CPI ID）调用 `fetch_bls_data`，而服务器从未返回过该 ID。Codex 写出的答案是真实的，因为
它告诉用户该数据不可用，但这次没有依据的获取操作导致用例失败。

就业用例的失败方式则不同。
Codex 可能会在调用 `resolve` 之前将请求收窄为某一个指标，
可能会在之后从并列的候选项中挑选一个，也可能两个都获取。
具体路径每次运行都可能不同，但每一种都会检索至少一个序列，
而不是停下来请求澄清，因此评分器判定其失败。

服务器可以向 Codex 提供缺失的指令。MCP 服务器会在
[握手](https://modelcontextprotocol.io/specification/2025-06-18/basic/lifecycle)期间发送一个
`instructions` 字段，规范将其描述为客户端可以添加到系统提示中的提示信息。Codex 会将它
传递给模型。独立循环使用自己的系统提示，会忽略该字段。

示例中该字段留空，因此新的运行仍然可能出现这些失败。在
`FastMCP` 构造函数上加一个字符串就能提供这一指引：

```python
GUIDANCE = """Answer only from this catalog. Call resolve first, passing the indicator the user
asked about without narrowing it. Pass fetch_bls_data only the series IDs that resolve returned,
never one from your own knowledge. If resolve returns no candidate, say the data is unavailable. If
several candidates tie for the best confidence, do not call fetch_bls_data at all. Name them and ask
which one the user wants."""

mcp = FastMCP("BLS evaluation fixture", instructions=GUIDANCE)
```

编辑 `server.py` 之后，请重启 MCP 服务器并重新运行 Codex 评估。服务器只在其
`initialize` 响应中发送 `instructions`，而正在运行的服务器会继续发送旧的、
空的值。

在带有指令的录制运行（`data/l3-trace-after.json`）中，Codex 的全部五个用例都
通过了，而工具和解析器都没有任何改动。

在录制的运行中，每个用例在 Codex 下耗时约 10 秒，在独立循环中耗时约 25 秒。



### 回放三次运行

三次录制的运行都位于 `data/` 中。评分器无需 API
密钥、服务器或 Codex 登录即可重新评估它们。带有指令的那次运行始终从其录制结果加载，
因为要实时复现它，就需要应用代码改动并重新运行 Codex 单元格。
`replay.py` 会对全部三次运行重新评分，`make replay` 则可以在终端中运行它：

In [10]:
%run replay.py

Standalone loop: 5/5 from data/l2-trace.json
Codex, empty instructions: 3/5 from data/l3-trace-before.json
    consumer-price-index: fetched series outside the expectation: ['CUUR0000SA0']
    employment-ambiguous: fetched series outside the expectation: ['CES0000000001', 'LNS12000000']
Codex, with instructions: 5/5 from data/l3-trace-after.json


notebook 会停止它所启动的服务器：

In [11]:
stop_server()

### 对照 ChatGPT 桌面应用进行抽查

本文中所有产品 harness 的数据都来自 OpenAI Codex SDK provider，而不是
ChatGPT 桌面应用本身。对本示例进行的一次健全性检查显示，其工具调用与 Codex
评估相同，包括推断出的 `CUUR0000SA0` ID。在实际项目中，全面的检查应覆盖更广泛的示例。

![ChatGPT 桌面应用中的 CPI](../../../images/harness-aware-desktop-cpi.png)

*在 ChatGPT 桌面应用中手动运行的 consumer price index 用例。*



## 解读结果：模型与 harness 的兼容性

示例插件的 CPI 用例以小规模展示了模型与 harness 的兼容性问题：同样的五个
问题在独立循环和 Codex 下产生了不同的结果。带有实时
API 调用和更多工具的生产版插件，其范围更具代表性，评估涵盖 56 项 BLS 调查中的 108 个问题。
基准测试结果来自这个包含 108 个问题的语料，并使用 LLM
评分标准评判器，而不是五用例示例中的工具调用检查。

| 设置项             | 取值                                                                                                                   |
| ------------------- | ----------------------------------------------------------------------------------------------------------------------- |
| 数据集             | 生产语料中预留的测试划分，涵盖 56 项 BLS 调查的 108 个单轮问题                           |
| 评判器               | 针对最终答案的 LLM 评分标准，`gpt-5.6-sol`                                                                         |
| 版本            | Codex 0.159.2，promptfoo 0.123.1                                                                                        |
| 重复次数             | 表 1 每个模型运行一次，表 2 和表 3 运行三次，且禁用响应缓存                         |

*生产基准测试配置。*



### 三个模型，一个 harness

三个模型在产品 harness Codex 下运行该语料。
模型是唯一的变量：

| 模型        | 通过率 | 每题工具调用次数 | 每题推理 token 数 |
| ------------ | --------- | --------------------- | --------------------------- |
| GPT-6.1 Sol  | 0.94      | 4.02                  | 19                          |
| GPT-6 Astra  | 0.94      | 3.74                  | 14                          |
| GPT-6 Luna   | 0.88      | 5.54                  | 1,229                       |

*表 1：生产版插件在 Codex 下、基于包含 108 个问题的
主要评估数据集的通过率和成本，所有模型使用同一个评分器。*

在本次运行中，GPT-6.1 Sol 和 GPT-6 Astra 在语料的 108 个问题中仅相差一题。这些
结果提供的证据太少，不足以可靠地对二者进行排名。

GPT-6 Luna 的得分低于更大的模型，
同时每题的工具调用次数比 GPT-6.1 Sol 多约 40%，推理 token 则是其 60 多倍。



### 观察到的差距变化

模型之间的差距还取决于产生它的 harness。GPT-5-mini 是一个旧版模型，之所以保留它作为
对比，是因为它是开发该插件时所针对的低成本基线。它在 Codex 下的表现下滑
促成了本文中的 harness 对比。GPT-6 Luna 和 GPT-5-mini
在 Codex 下相差甚远，而在独立循环中则接近得多：

| Harness                                      | GPT-6 Luna   | GPT-5-mini |
| -------------------------------------------- | ------------ | ---------- |
| 产品 harness（Codex）                      | 0.88         | 0.43       |
| 独立函数调用循环             | 0.74         | 0.64       |

*表 2：生产版插件在两种 harness 下的表现，问题与表 1 相同。*

在独立循环中，GPT-6 Luna 和 GPT-5-mini 平均相差约 10 个百分点，单次运行中相差 1 到
19 个百分点。在 Codex 下，差距要大得多。与独立循环相比，GPT-6 Luna 提升了 14 个百分点，
GPT-5-mini 则下降了 21 个百分点。



### 为什么 GPT-5-mini 在 Codex 下表现更差

GPT-5-mini 在那里的失败中，大约有一半与 shell 有关。模型会调用智能体的
shell 并运行一些毫无作用的命令。它大约一半的答案以向用户提问结束。
在整个语料中，它在 Codex 下大约每 100 个问题中有 85 个会调用工具，而在同一 harness 上，
GPT-6 Luna 几乎每个问题都会调用工具。独立循环中没有 shell。在
完全相同的语料上，GPT-5-mini 在那里大约每 100 个问题中有 95 个会调用工具。



### 为什么 GPT-6 Luna 在独立循环中表现更差

GPT-6 Luna 失败的原因不同。在独立循环中，它的大多数失败答案要么声称
所请求的数据不可用，要么在达到循环的步数上限时终止。
在 Codex 下，这些轮次会继续进行，这或许可以解释那里每题
多出的大部分工具调用，尽管这些运行并未单独验证这一机制。



## 考虑非确定性

请将本文中的通过率视为相对指标。独立循环与 Codex 之间的通过率相差 14 和 21 个百分点。下面的重复实验说明了结果在不同运行之间可能如何变化。

直接工具测试每次都应返回相同的结论，因为它们运行时没有模型参与。
然而，涉及模型的运行，即使配置相同，不同运行之间也可能产生不同的结果：

- 模型可能会选择另一条路径来完成任务。
- LLM 评判器可能会对同一个答案给出不同的评估。
- 差异也可能由模型或 harness 的外部更新引起。

使用 GPT-6 Luna、以相同的 Codex 配置，在上述基准测试配置的 108 个问题上
运行三次，得到如下分布：

| 重复                      | 1     | 2     | 3     |
| --------------------------- | ----- | ----- | ----- |
| 通过率                   | 0.88  | 0.89  | 0.86  |
| 每题工具调用次数       | 5.54  | 5.70  | 5.20  |
| 每题推理 token 数 | 1,229 | 1,195 | 1,146 |

*表 3：生产版插件在 Codex 下的三次重复运行。*

- 整体通过率几乎没有变化：三次运行之间的波动为 3 个百分点。
- 单个问题的变化远比这一波动所暗示的要大。108 个问题中有 23 个（21%）在三次运行中
  的某次在通过和失败之间翻转，而总体结果保持稳定。
- 工具调用的波动同样剧烈：某个问题在一次重复中用了 5 次工具调用，在另一次中则用了 19 次。



## 实用要点

使用能够暴露你正在调查的问题的最低成本方法：

- 每当解析器逻辑或 fixture 数据发生变化时，运行直接工具测试。
- 在迭代工具描述、提示、评分或模型选择时，使用独立循环。
- 对每个新模型都运行产品 harness，而不是假设两种方法的结论一致。如果
  二者确实一致，就在独立循环中迭代，并将产品 harness 留给发布检查以及
  面向利益相关者的结果。

编写那些看似合理的答案却是错误的测试用例，例如正确答案是一个简短的
序列列表、一次澄清请求（“价格在上涨吗？”可能指消费者价格，也可能指生产者价格），或者根本没有
数据。像示例中的 `mechanism` 和 `expected_type` 那样，为每个用例标注它测试的内容以及什么算作正确。
这样就可以将一次运行过滤到正在开发的那一部分，
同样的标签也能显示哪些领域最需要新的测试用例。

将评估 trace 与生产运行进行比较，以发现配置遗漏的差异，例如
智能体指令、工具可用性或运行时版本。只测量一次的通过率，看起来也可能
比其背后逐个用例的真实情况稳定得多。
自动重复运行相同的用例来衡量它们的稳定性，并将这种
稳定性本身作为一项指标。一个对同一问题的结论在不同运行间摇摆不定的系统，
可能不如一个平均分略低但行为更可预测的系统有用。

更便宜的 harness 和更小的模型有助于快速迭代，但它们只能
近似最终发布的形态。生产环境的行为来自所部署的 harness
与模型的组合。在信任最终那次运行之前，需要确认以下几点：

- 模型在评估中看到的框架与在生产环境中看到的相同，并且每一个本应传递给它的配置值
  都确实传递到了。
- 同一配置在关闭缓存的情况下运行过不止一次。
- 在查看通过率的同时，也检查了每题的工具调用次数和推理 token 数。
- 有人工审阅过具有代表性的答案样本。



### 亲自尝试

多层级结构以及共享同一数据集和评分器的模式是可复用的，
但要将它们适配到其他插件，需要修改各个组件：

| 领域                 | 文件                                                                 | 操作  | 需要修改的内容                                                                                               |
| -------------------- | --------------------------------------------------------------------- | ------- | ------------------------------------------------------------------------------------------------------------ |
| MCP 服务器           | `server.py`                                                         | 替换 | 使用目标服务器，并更新独立循环和插件所使用的 MCP URL。                |
| 测试用例           | `promptfooconfig.yaml`                                              | 更新  | 提供特定领域的查询、结构化输入、期望和元数据。                               |
| 评估逻辑     | `level_one.py`、`eval_grading.py`                                 | 更新  | 使直接工具测试和共享评分适配目标工具的 schema 和期望。评分器会检查工具名称、参数格式和预期结果字段，因此当这些发生变化时，评分器也必须随之修改。 |
| 指令         | `provider.py` 以及 MCP 服务器的 `instructions`                 | 更新  | 使独立循环的系统提示以及产品可见的指引与预期行为保持一致。产品 harness 通常会提供自己的系统提示，因此针对它的指引应放在工具描述或 MCP 服务器的 instructions 中。 |
| 插件配置 | `.codex-home/marketplaces/`、`.mcp.json`、`config.toml.example` | 更新  | 设置插件标识、marketplace 位置、MCP 端点以及隔离的 Codex home。                        |
| 评估结构 | 全部三种方法                                                  | 保留  | 保持直接工具测试、独立循环和产品 harness 之间的分离。                    |



## 延伸阅读

- [AI Agents That Matter](https://arxiv.org/abs/2407.01502)
  解释了评估智能体与评估模型有何不同。
- [τ-bench](https://arxiv.org/abs/2406.12045)
  引入了 `pass^k`，即 `k` 次独立尝试全部成功的概率。
- [Moving from OpenAI Evals to Promptfoo](https://developers.openai.com/cookbook/examples/evaluation/moving-from-openai-evals-to-promptfoo)
  逐个用例地在 promptfoo 中重建了一套 OpenAI Evals 测试。
- [Codex sandboxing](https://developers.openai.com/codex/concepts/sandboxing)
  介绍了 Codex 运行可以采用的各种模式以及每种模式允许的操作。



## 贡献者

本 cookbook 由 OpenAI 与 [deepsense.ai](https://deepsense.ai/) 合作完成，deepsense.ai 构建并评估了 BLS 连接器。

- [Maciej Domagała](https://www.linkedin.com/in/macdomagala/)
- [Łukasz Dragan](https://www.linkedin.com/in/%C5%82ukasz-d-a22b3986/)
- [Michał Rdzany](https://www.linkedin.com/in/micha%C5%82-rdzany-85567a225/)
- [Danny Wigg](https://www.linkedin.com/in/dannywigg/)